# CoGAPS

In [1]:
library(CoGAPS)
library(here)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Input

In [2]:
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))
K <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))
K <- as.integer(K)

cat("Dimensions:", dim(gtex_data), "\n")
cat("nPatterns (K):", K, "\n")

Dimensions: 21613 17382 
nPatterns (K): 412 


In [3]:
gtex_data_shifted <- gtex_data - min(gtex_data)

## Distributed CoGAPS run

In [ ]:
# Calculate nSets: each subset should have 1000-5000 genes
n_genes <- nrow(gtex_data_shifted)
nSets <- ceiling(n_genes / 2500)
cat("Number of genes:", n_genes, "\n")
cat("nSets (subsets):", nSets, "\n")

params <- CogapsParams(
    nPatterns = K,
    nIterations = 5000,
    seed = 123,
    distributed = "genome-wide"
)
params <- setDistributedParams(params, nSets = nSets)

In [5]:
cogapsresult <- CoGAPS(gtex_data_shifted, params, nThreads = 4, outputFrequency = 10000)

Warning message in CoGAPS(gtex_data_shifted, params, nThreads = 4, outputFrequency = 10000):
“requesting multi-threaded version of CoGAPS but compiler did not support OpenMP”
Warning message in checkInputs(data, uncertainty, allParams):
“running distributed cogaps without mtx/tsv/csv/gct data”


## Output

In [ ]:
B <- t(cogapsresult@sampleFactors)  # K x samples

sample_names <- colnames(gtex_data)

rownames(B) <- paste0("LV", seq_len(nrow(B)))
colnames(B) <- sample_names

head(B)
dim(B)

In [ ]:
output_dir <- here("output/gtex/cogaps")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(
  B,
  file = file.path(output_dir, "gtex_B.csv"),
  quote = FALSE
)

In [ ]:
saveRDS(cogapsresult, file.path(output_dir, "cogaps_model.rds"))